In [ ]:
from pathlib import Path
from typing import Union, Sequence, Optional, Tuple

import numpy as np
import cv2
import matplotlib.pyplot as plt


def load_gray(path: Union[str, Path]) -> np.ndarray:
    """
    Load an image as float32 grayscale array (0-255).
    """
    p = Path(path)
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(f"Could not read image: {p}")
    return img.astype(np.float32)


def crop_image(img: np.ndarray, crop_box: Tuple[int, int, int, int]) -> np.ndarray:
    """Crop an image given (x, y, w, h)."""
    x, y, w, h = crop_box
    return img[y:y + h, x:x + w]


def select_crop_region(image_path: Union[str, Path]) -> Tuple[int, int, int, int]:
    """
    Opens an interactive OpenCV window to crop the baseline image.
    If a directory is passed, the first image in that directory is used.
    Returns (x, y, w, h) of the crop.
    """
    p = Path(image_path)
    if p.is_dir():
        images = sorted(
            [q for q in p.iterdir() if q.suffix.lower() in {".tif", ".tiff", ".png", ".jpg", ".jpeg"}]
        )
        if not images:
            raise ValueError(f"No images found in baseline dir: {p}")
        image_path = images[0]
    else:
        image_path = p

    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(image_path)

    cv2.namedWindow("Crop Baseline", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Crop Baseline", 1200, 800)

    print("Select crop region and press ENTER or SPACE when done. Press ESC to cancel.")
    r = cv2.selectROI("Crop Baseline", img, showCrosshair=True)
    cv2.destroyAllWindows()

    x, y, w, h = map(int, r)
    if w == 0 or h == 0:
        raise ValueError("No region selected.")
    print(f"Crop selected: x={x}, y={y}, w={w}, h={h}")
    return x, y, w, h


def gather_frames(folder: Union[str, Path], pattern: str = "*.tif") -> Sequence[Path]:
    folder = Path(folder)
    frames = sorted(folder.glob(pattern))
    if not frames:
        raise ValueError(f"No frames found in {folder} matching {pattern}")
    return frames


def compute_mean_intensity_sequence(
    folder: Union[str, Path],
    crop_box: Optional[Tuple[int, int, int, int]] = None,
    pattern: str = "*.tif"
) -> np.ndarray:
    """
    Compute mean grayscale intensity for each frame in a folder.
    Returns a 1D array of length N_frames.
    """
    frames = gather_frames(folder, pattern=pattern)
    means = []

    for fp in frames:
        img = load_gray(fp)
        if crop_box is not None:
            img = crop_image(img, crop_box)
        means.append(img.mean())

    return np.array(means, dtype=np.float32)


def plot_light_fluctuations(
    folder: Union[str, Path],
    fps: float,
    n_frames: Optional[int] = None,
    crop_box: Optional[Tuple[int, int, int, int]] = None,
    pattern: str = "*.tif",
    title: Optional[str] = None
) -> np.ndarray:
    """
    Compute and plot mean grayscale intensity over time.

    Parameters
    ----------
    folder : folder containing frames
    fps : frames per second (must be > 0)
    n_frames : optional number of frames to use for the time axis
        - If None, uses however many frames are found in the folder.
        - If provided and differs from frames found, the plot uses the
          computed intensities length, and time is based on that length
          (with a warning-like behavior via ValueError only if n_frames < 1).
    """
    if fps <= 0:
        raise ValueError("fps must be > 0")

    intensities = compute_mean_intensity_sequence(folder, crop_box=crop_box, pattern=pattern)

    # Decide how many frames define the time axis
    N = len(intensities) if n_frames is None else int(n_frames)
    if N < 1:
        raise ValueError("n_frames must be >= 1")

    if N != len(intensities):
        N = len(intensities)

    time_s = np.arange(N, dtype=np.float32) / float(fps)

    plt.figure(figsize=(8, 4))
    plt.plot(time_s, intensities[:N])
    plt.xlabel("Time (s)")
    plt.ylabel("Mean grayscale intensity")
    if title is None:
        title = f"Light fluctuations in {Path(folder).name}"
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return intensities


In [ ]:
baseline_1 = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_background"

# This uses the first image in the folder to pick a crop
crop_box = select_crop_region(baseline_1)

# Now analyze that folder using the chosen crop
intensities = plot_light_fluctuations(baseline_1, fps=1000000, n_frames=180, crop_box=crop_box)


In [ ]:
baseline_2 = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_background"
crop_box2 = select_crop_region(baseline_2)
intensities = plot_light_fluctuations(baseline_2, fps=1000000, n_frames=180, crop_box=crop_box2)


In [ ]:
baseline_3 = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_canned_air PROCESSED (paired option 1)"
intensities = plot_light_fluctuations(baseline_3, fps=1000000, n_frames=180, crop_box=None)

In [ ]:
baseline_4 = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_canned_air PROCESSED (paired option 3 cropped alpha)"
intensities = plot_light_fluctuations(baseline_4, fps=1000000, n_frames=180, crop_box=None)


In [ ]:
baseline_5 = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_canned_air PROCESSED (paired)"
intensities = plot_light_fluctuations(baseline_5, fps=1000000, n_frames=180, crop_box=None)

In [ ]:
not_paired = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260311\260311_canned_air PROCESSED (paired option 3)"

intensities = plot_light_fluctuations(not_paired, fps=1000000, n_frames=180, crop_box=None)

In [ ]:
paired = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260219\260219010 PROCESSED"

intensities = plot_light_fluctuations(paired, fps=1000000, n_frames=180, crop_box=None)

In [ ]:
paired_option1 = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260219\260219010 PROCESSED (paired option 1)"
intensities = plot_light_fluctuations(paired_option1, fps=1000000, n_frames=180, crop_box=None)

In [ ]:
paired_option2 = r"C:\Users\ahada\OneDrive\University of Washington\Z-Pinch Fusion\Schlieren post processing\Schlieren data\260219\260219010 PROCESSED (paired option 1)"
intensities = plot_light_fluctuations(paired_option2, fps=1000000, n_frames=180, crop_box=None)